# IsoCourt — VideoMAE+TimeSformer Training on FineBadminton-20K

Registry category: **`videomae_timesformer`** | Script: `train_videomae_timesformer.py` | Checkpoint: `badminton_model_videomae_timesformer.pth`

**Runtime → Change runtime type → GPU (A100 recommended, T4 works)**

In [ ]:
import torch, os
assert torch.cuda.is_available(), 'No GPU — change runtime to GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/IsoCourt/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)

## 2 — Clone repo & install dependencies

In [ ]:
REPO = 'https://github.com/Navneethd8/IsoCourt.git'
BRANCH = 'main'

if not os.path.isdir('/content/IsoCourt'):
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 -b {BRANCH} {REPO} /content/IsoCourt
else:
    !cd /content/IsoCourt && git pull --ff-only

%cd /content/IsoCourt

In [ ]:
!pip install -q opencv-python-headless mediapipe mlflow tqdm timm transformers safetensors huggingface_hub

## 3 — Download FineBadminton-20K & prepare data

In [ ]:
DATA_DIR = '/content/IsoCourt/backend/data/FineBadminton-20K'
MERGED_JSON = '/content/IsoCourt/backend/data/transformed_combined_rounds_output_en_evals_translated.json'

if not os.path.isfile(MERGED_JSON):
    !python backend/pipelines/vlm/common/prepare_finebadminton_20k.py \
        --local-dir {DATA_DIR} --max-workers 8
else:
    print(f'Merged JSON exists: {MERGED_JSON}')

## 4 — Train VideoMAE+TimeSformer

In [ ]:
EPOCHS = 30
SAVE_PATH = os.path.join(DRIVE_CKPT, 'badminton_model_videomae_timesformer.pth')
POSE_CACHE = os.path.join(DRIVE_CKPT, 'pose_cache_staeformer.pt')
USE_POSE = True

import sys
sys.path.insert(0, '/content/IsoCourt/backend')

from pipelines.training.train_videomae_timesformer import train_videomae_timesformer

train_videomae_timesformer(
    data_root='/content/IsoCourt/backend/data',
    list_file=MERGED_JSON,
    epochs=EPOCHS,
    batch_size=4,
    lr=1e-4,
    device='cuda',
    save_path=SAVE_PATH,
    pose_cache_path=POSE_CACHE,
    hf_model_id='MCG-NJU/videomae-base',
    freeze_videomae=True,
    videomae_unfreeze_last_n=0,
    lr_mult=5.0,
    videomae_lr_mult=0.1,
    weight_decay=1e-2,
    label_smoothing=0.1,
    scheduler_t0=10,
    scheduler_t_mult=2,
    accumulation_steps=4,
    stroke_loss_weight=2.0,
    aug_strength='strong',
    embed_dim=128,
    depth=4,
    num_heads=4,
    checkpoint_metric='val_type_acc',
    use_pose=USE_POSE,
)

In [ ]:
print('Training complete.')
!ls -lh {DRIVE_CKPT}/*videomae_timesformer* 2>/dev/null || echo 'No checkpoints found'